In [ ]:
from scipy import stats
from statsmodels.stats.oneway import anova_oneway
import seaborn as sns
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt

In [ ]:
# --- Beispiel mit Pinguine ---
penguins = sns.load_dataset('penguins').dropna()
plt.figure(figsize=(10, 6))
sns.boxplot(data = penguins, y = "flipper_length_mm", x = "species")
plt.tight_layout()
plt.savefig("boxplot1")
plt.show()

In [ ]:
def run_smart_anova(df, dependent_var, grouping_var):
    print(f"--- Analyse: {dependent_var} nach {grouping_var} ---")

    # 1. Daten vorbereiten
    data = df[[dependent_var, grouping_var]].dropna()

    # Gruppen für die Tests vorbereiten
    group_names = data[grouping_var].unique()
    groups = [data[data[grouping_var] == name][dependent_var] for name in group_names]

    # 2. Check: Normalverteilung (Shapiro-Wilk) oder n > 30
    is_normal = True
    for name in group_names:
        group_data = data[data[grouping_var] == name][dependent_var]
        n = len(group_data)
        if n < 30:
            _, p_shapiro = stats.shapiro(group_data)
            if p_shapiro < 0.05:
                is_normal = False
                print(f"ACHTUNG: Gruppe {name} ist nicht normalverteilt (p={p_shapiro:.4f}).")

    if is_normal or all(len(g) > 30 for g in groups):
        print("Check: Normalverteilung ok (oder n > 30).")
    else:
        print("Check: Voraussetzungen für ANOVA kritisch (Nicht normal & n klein).")

    # 3. Check: Homoskedastizität (Levene-Test)
    _, p_levene = stats.levene(*groups)
    print(f"Levene-Test (p-Wert): {p_levene:.4f}")

    # 4. Durchführung der ANOVA via statsmodels
    if p_levene > 0.05:
        print("Entscheidung: Nutze Standard-ANOVA (Varianzhomogenität ok).")
        # use_var='equal' entspricht der klassischen Fisher-ANOVA
        res = anova_oneway(data[dependent_var], data[grouping_var], use_var='equal')
    else:
        print("Entscheidung: Nutze Welch-ANOVA (Varianzhomogenität verletzt!).")
        # use_var='unequal' entspricht der Welch-ANOVA
        res = anova_oneway(data[dependent_var], data[grouping_var], use_var='unequal')

    print("ANOVA Tabelle:")
    for name in ["statistic", "pvalue", "df", "df_num", "df_denom", "nobs_total"]:
        print(f"{name} = {getattr(res, name)}")

    return res


# Wir untersuchen: Unterscheidet sich die Flossenlänge je nach Art?
res = run_smart_anova(penguins, 'flipper_length_mm', 'species')


### Aufgabe: Einfluss der Passagierklasse auf das Alter (Titanic-Datensatz)

Untersuche mit einer ANOVA, ob die Passagierklasse (Pclass) einen Einfluss auf das Alter der Passagiere (Age) hat.

Prüfe die Voraussetzungen für eine ANOVA:
* Normalverteilung innerhalb der Gruppen (oder ausreichend große Stichproben)
* Varianzhomogenität mit dem Levene-Test
* Entscheide, ob eine Standard-ANOVA oder eine Welch-ANOVA verwendet werden sollte.

Führe die Analyse durch.
* Interpretiere die Ergebnisse (insbesondere den p-Wert).
* Formuliere ein Fazit zur Fragestellung:

Unterscheidet sich das durchschnittliche Alter der Passagiere zwischen der 1., 2. und 3. Klasse signifikant?

In [ ]:
# --- Beispiel mit Titanic ---
titanic = sns.load_dataset('titanic')
sns.boxplot(data = titanic, y = "age", x = "pclass")

In [ ]:
results = run_smart_anova(titanic, 'age', 'pclass')

alpha = 0.05

if results.pvalue < alpha:
    print(
        f"\nErgebnis: p = {results.pvalue:.4g} < {alpha}. "
        "Die Nullhypothese wird verworfen. "
        "Die Passagierklasse (pclass) hat einen statistisch signifikanten Einfluss auf das Alter (age)."
    )
else:
    print(
        f"\nErgebnis: p = {results.pvalue:.4g} >= {alpha}. "
        "Die Nullhypothese kann nicht verworfen werden. "
        "Es gibt keinen statistisch signifikanten Hinweis darauf, dass die Passagierklasse (pclass) das Alter (age) beeinflusst."
    )

#### Die Lösung: Post-Hoc-Tests (Tukey HSD)

Um herauszufinden, welche Paare sich genau unterscheiden, brauchen wir einen Post-Hoc-Test. Der Goldstandard ist hier der Tukey-HSD-Test (Honestly Significant Difference).

Warum machen wir nicht einfach drei normale t-Tests?
Wenn du viele t-Tests hintereinander machst, summiert sich dein Fehlerrisiko (die sogenannte Alpha-Fehler-Kumulierung). Tukey korrigiert das mathematisch, sodass dein Gesamtrisiko bei 5 % bleibt.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 'data' ist dein DataFrame, 'species' die Gruppen, 'flipper_length_mm' die Werte
tukey = pairwise_tukeyhsd(endog=penguins['flipper_length_mm'],
                          groups=penguins['species'],
                          alpha=0.05)

print(tukey)

# Optional: Die Ergebnisse visualisieren
# tukey.plot_simultaneous()

#### Der Paar-Vergleich (1 gegen 1)

Stell dir vor, du hast drei Spezies: Adelie, Chinstrap und Gentoo. Der Tukey-Test erstellt drei Zeilen:

* Adelie vs. Chinstrap: Ist reject=True? -> Dann sind diese beiden verschieden.

* Adelie vs. Gentoo: Ist reject=True? -> Dann sind diese beiden verschieden.

* Chinstrap vs. Gentoo: Ist reject=True? -> Dann sind diese beiden verschieden.

In [ ]:
# --- Beispiel mit Tips, Two-Way ---
tips = sns.load_dataset("tips")

plt.figure(figsize=(10, 6))
sns.boxplot(data = tips, y = "tip", x = "day", hue ="smoker")
plt.tight_layout()
plt.savefig("boxplot2")
plt.show()

#### Leitfaden: Two-Way ANOVA Modell erstellen

Die zweifaktorielle ANOVA (Two-Way ANOVA) untersucht den Einfluss von zwei kategorialen unabhängigen Variablen auf eine metrische abhängige Variable. Sie ist das Standardwerkzeug, wenn man wissen möchte, ob zwei Faktoren gleichzeitig (und in ihrem Zusammenspiel) ein Ergebnis beeinflussen.

---

##### 🛠️ Vertiefung: Die Patsy-Formelsprache (Modell-Syntax)

In Python (`statsmodels`) nutzen wir die Patsy-Syntax, um statistische Modelle zu definieren. Sie ist extrem mächtig, da sie Transformationen direkt in der Formel erlaubt.

---

##### 1. Die Logik der Operatoren

* **Die Tilde `~`**: Trennt Zielvariable (links) von den Einflussgrößen (rechts).
* **Das Plus `+`**: Fügt Variablen zum Modell hinzu (additiv). Beispiel: `y ~ A + B` (Haupteffekte von A und B).
* **Der Doppelpunkt `:`**: Berechnet **nur** die Wechselwirkung (Interaktion). Beispiel: `y ~ A:B`.
    * Ist der Unterschied zwischen Rauchern und Nichtrauchern an allen Tagen gleich, oder ändert er sich je nach Tag?
* **Der Stern `*`**: Die Kurzschreibweise für Haupteffekte **und** Wechselwirkung.
    * `y ~ A * B` ist exakt das Gleiche wie `y ~ A + B + A:B`.
    * **Wichtig:** Schreibe niemals `A + B + A * B`, da der Stern die Haupteffekte bereits beinhaltet (Redundanz).

---

##### 2. Spezial-Funktionen und Operatoren

* **`C()` (Categorical)**: Markiert eine Variable als Faktor. Das ist essenziell, damit Python z. B. Wochentage nicht als Zahlen (1, 2, 3) behandelt, sondern als separate Kategorien.
* **`I()` (Identity-Funktion)**: Wird genutzt, um mathematische Operationen *innerhalb* der Formel zu schützen.
    * Da Symbole wie `+`, `-`, `*` in Patsy eine statistische Bedeutung haben, kannst du mit `I()` echtes Rechnen erzwingen.
    * Beispiel: `y ~ x1 + I(x1**2)` (fügt ein quadratisches Glied hinzu).
* **`- 1` oder `+ 0`**: Entfernt den Intercept (den Achsenabschnitt). Das Modell wird gezwungen, durch den Nullpunkt zu gehen. Beispiel: `y ~ x - 1`.
* **`np.log()` / `np.sqrt()`**: Du kannst Standard-Funktionen direkt auf Variablen anwenden. Beispiel: `np.log(y) ~ x`.

---

##### 3. Beispiel-Vergleich für dein Notebook

| Formel | Was wird berechnet? |
| :--- | :--- |
| `tip ~ day + smoker` | Wirkt der Tag **und** wirkt das Rauchen? (Kein Zusammenspiel). |
| `tip ~ day : smoker` | **Nur** die Wechselwirkung (sehr selten sinnvoll ohne Haupteffekte). |
| `tip ~ day * smoker` | Wirkt der Tag, wirkt das Rauchen **und** gibt es ein spezielles Zusammenspiel? |
| `tip ~ total_bill + I(total_bill**2)` | Ein Modell, das prüft, ob das Trinkgeld quadratisch mit der Rechnung steigt. |

---

##### 💡 Warum wir meistens `*` nutzen
In der ANOVA interessiert uns fast immer das gesamte Bild. Wenn wir nur `A:B` (Wechselwirkung) ohne die Haupteffekte `A` und `B` rechnen würden, könnten wir nicht sicher sein, ob der Effekt wirklich vom Zusammenspiel kommt oder nur von einem der Einzelfaktoren. Der Stern `*` ist also der "Sicherheits-Standard".


In [ ]:
# Vorab-Check: Levene-Test für ALLE Kombinationen (Tag + Raucher)
# Wir gruppieren nach beiden Spalten, um die Varianz in jeder "Zelle" zu prüfen
grouped_data = [group['tip'].values for name, group in tips.groupby(['day', 'smoker'])]
stat, p_levene = stats.levene(*grouped_data)
print(f"Levene-Test p-Wert: {p_levene:.4f}")

# Modell erstellen
# 'tip ~ day * smoker' berechnet Haupt- und Wechselwirkungen
model = ols('tip ~ C(day) * C(smoker)', data=tips).fit()

# Entscheidung mit if: Falls p < 0.05, nutze robuste Standardfehler (hc3)
if p_levene > 0.05:
    print("Varianzen sind gleichmäßig (Homoskedastizität) -> Standard ANOVA")
    anova_results = anova_lm(model, typ=2)
else:
    print("Varianzen sind ungleichmäßig -> Nutze robuste ANOVA (HC3)")
    anova_results = anova_lm(model, typ=2, robust='hc3')

print("Zweifaktorielle ANOVA erfolgreich:")
print(anova_results)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Daten laden
tips = sns.load_dataset("tips")

plt.figure(figsize=(10, 6))

# Pointplot zeigt die Mittelwerte und deren Konfidenzintervalle
sns.pointplot(data=tips, x='day', y='tip', hue='smoker',
              dodge=True, # Verschiebt die Punkte leicht, damit sie sich nicht überlappen
              capsize=.1, # Fügt kleine Kappen an die Fehlerbalken
              markers=["o", "s"], linestyles=["-", "--"])

plt.title("Trinkgeld-Trends", fontsize=14)
plt.ylabel("Durchschnittliches Trinkgeld ($)")
plt.xlabel("Wochentag")
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("interaction_plot.png")
plt.show()

#### 🔀 Post-Hoc-Analyse für Two-Way ANOVA

Die Two-Way ANOVA (z. B. `day * smoker`) sagt dir nur, **ob** es irgendwo
Unterschiede gibt, aber nicht **wo**. Dafür brauchst du einen Post-hoc-Test.

Bevor du loslegst, schau dir in der ANOVA-Tabelle die **Interaktion**
(`day:smoker`) an. Sie beantwortet die Frage: Hängt der Effekt des Tages
davon ab, ob jemand raucht? Davon hängt ab, wie du weitermachst.
-> Interaktion day:smoker (p = 0,54): Der Effekt des Tages hängt nicht davon ab, ob jemand raucht. Du bist also in Fall 1.

---

#### Fall 1: Interaktion nicht signifikant → Haupteffekte anschauen

Die Faktoren wirken unabhängig voneinander. Der Unterschied zwischen den Tagen
ist also für Raucher und Nichtraucher ungefähr gleich. Deshalb kannst du jeden
Faktor einzeln betrachten:

* **`day`** hat 4 Stufen → Tukey-Test über die Tage, um zu sehen, welche
  Tage sich unterscheiden.
* **`smoker`** hat nur 2 Stufen → kein Post-hoc-Test nötig. Ist der Effekt
  in der ANOVA signifikant, weißt du schon, dass sich die beiden Gruppen
  unterscheiden.

---

#### Fall 2: Interaktion signifikant → Einfache Haupteffekte

Jetzt verhalten sich Raucher und Nichtraucher über die Woche **unterschiedlich**.
Eine Aussage wie "Am Samstag wird mehr Trinkgeld gegeben" wäre dann zu pauschal,
weil sie vielleicht nur für eine der beiden Gruppen stimmt.

Deshalb vergleichst du die Tage **getrennt innerhalb jeder Gruppe**:
* Unterscheiden sich die Tage bei den Rauchern?
* Unterscheiden sich die Tage bei den Nichtrauchern?

Wichtig: Weil du so mehrere Tests machst, müssen die p-Werte korrigiert werden
(z. B. mit der Holm-Korrektur), sonst findest du zufällig "Unterschiede", die
es gar nicht gibt. `pingouin` erledigt das in einem Schritt:

    pg.pairwise_tests(data=df, dv='tip', between=['smoker', 'day'], padjust='holm')

---

#### Alternative: Die "Flattening"-Technik

Man kann beide Faktoren auch zu einer Spalte verschmelzen (aus "Samstag" und
"Raucher" wird "Sat_Yes") und dann einen normalen Tukey-Test über alle
Kombinationen rechnen.

Das klingt praktisch, hat aber Nachteile:
* **Sehr viele Vergleiche:** 4 Tage × 2 Gruppen = 8 Kombinationen → 28 Tests.
  Je mehr Tests, desto strenger die Korrektur und desto schwerer wird es,
  echte Unterschiede zu finden.
* **Schwer interpretierbar:** Ein Vergleich wie "Raucher am Samstag" vs.
  "Nichtraucher am Sonntag" unterscheidet sich in **beiden** Faktoren. Wenn er
  signifikant ist, weißt du nicht, ob es am Tag oder am Rauchen liegt.

Diese Methode eignet sich daher höchstens, um sich einen ersten Überblick
zu verschaffen.

* day (p = 0,16): Das Trinkgeld unterscheidet sich nicht signifikant zwischen den Tagen.
* smoker (p = 0,68): Raucher und Nichtraucher geben nicht signifikant unterschiedlich viel Trinkgeld.

Ein Post-hoc-Test ist deshalb nicht nötig. Wenn die ANOVA keinen Unterschied findet, gibt es auch nichts, was man genauer lokalisieren müsste. Einen Tukey-Test könntest du technisch trotzdem rechnen, er sollte dann aber auch nichts finden, und man sollte ihn nicht als Suche nach “doch noch irgendwas” missbrauchen.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(endog=tips['tip'], groups=tips['day'], alpha=0.05)
print(tukey)

In [ ]:
# Flattening (alle 8 Kombinationen gegeneinander)
tips['day_smoker'] = tips['day'].astype(str) + '_' + tips['smoker'].astype(str)
print(pairwise_tukeyhsd(endog=tips['tip'], groups=tips['day_smoker'], alpha=0.05))